=== Regions in Disease but missing in other datasets ===
Missing in Climate: 2
Missing in Crop: 82
Missing in Livestock: 1
Missing in Landuse: 1

Missing in Climate:
  - ('Senegal', 'Dakar')
  - ('South Africa', 'Western Cape')

Missing in Landuse:
  - ('Cameroon', 'Extrême - Nord')

-> climate, landuse 데이터에서 없는 지역들이 가지는 disease 데이터의 year 특성이 2021-2023인지 체크하기
=== Year Range in Each Dataset ===
- Disease: 2001 - 2023
- Climate: 2001 - 2020
- Crop: 1974 - 2023
- Livestock: 2001 - 2021
- Landuse: 2015 (fixed)

## **라이브러리 및 데이터 로드**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

# 파일 경로 설정
climate_path = '../../../data/interim/climate/sorted_climate_data.csv'
crop_path = '../../../data/interim/crop/processed_crop.csv'
landuse_2015_path = '../../../data/interim/landuse/processed_landuse_2015.csv'
landuse_2019_path = '../../../data/interim/landuse/processed_landuse_2019.csv'  # 여기선 사용하지 않음
livestock_path = '../../../data/interim/livestock/processed_livestock.csv'
disease_path = '../../../data/interim/disease/processed_HIV_rates.csv'

# 데이터 로드
print("Loading datasets...")
climate_df = pd.read_csv(climate_path)
crop_df = pd.read_csv(crop_path)
landuse_2015 = pd.read_csv(landuse_2015_path)
landuse_2019 = pd.read_csv(landuse_2019_path)
livestock_df = pd.read_csv(livestock_path)
disease_df = pd.read_csv(disease_path)

print("✓ All datasets loaded successfully!")

Libraries imported successfully!
Loading datasets...
✓ All datasets loaded successfully!


## **데이터 병합하기**

In [2]:
# Disease (기준 데이터)
base_df = disease_df[['year', 'country', 'region_name', 'infection_rate']].copy()
base_df = base_df.rename(columns={'region_name': 'admin'})

print(f"Base (disease) columns: {base_df.columns.tolist()}")
print(f"Base shape: {base_df.shape}")

# Climate - 이미 country, admin, year 형식이라고 가정
# 만약 다른 이름이면 아래 주석을 해제하고 수정
climate_clean = climate_df.copy()
# climate_clean = climate_clean.rename(columns={'기존컬럼명': 'country', '기존컬럼명2': 'admin'})

climate_clean = climate_clean[['country', 'admin', 'year', 
                                'max_temperature', 'min_temperature', 'avg_temperature']]

print(f"Climate columns: {climate_clean.columns.tolist()}")
print(f"Climate shape: {climate_clean.shape}")

# Livestock
livestock_clean = livestock_df[['ADM0_NAME', 'ADM1_NAME', 'year',
                                 'Buffa', 'Cattl', 'Chick', 'Ducks', 
                                 'Goats', 'Horse', 'Sheep', 'Swine']].copy()

livestock_clean = livestock_clean.rename(columns={
    'ADM0_NAME': 'country', 
    'ADM1_NAME': 'admin'
})

print(f"Livestock columns: {livestock_clean.columns.tolist()}")
print(f"Livestock shape: {livestock_clean.shape}")

# Landuse - 2015년 데이터만 사용 (year 컬럼 제거)
landuse_clean = landuse_2015[['ADM0_NAME', 'ADM1_NAME', 'rate_landuse']].copy()

landuse_clean = landuse_clean.rename(columns={
    'ADM0_NAME': 'country', 
    'ADM1_NAME': 'admin'
})

print(f"Landuse columns: {landuse_clean.columns.tolist()}")
print(f"Landuse shape: {landuse_clean.shape}")
print("\nNote: Landuse 2015 data will be applied to all years")

Base (disease) columns: ['year', 'country', 'admin', 'infection_rate']
Base shape: (488, 4)
Climate columns: ['country', 'admin', 'year', 'max_temperature', 'min_temperature', 'avg_temperature']
Climate shape: (15600, 6)
Livestock columns: ['country', 'admin', 'year', 'Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine']
Livestock shape: (5796, 11)
Landuse columns: ['country', 'admin', 'rate_landuse']
Landuse shape: (859, 3)

Note: Landuse 2015 data will be applied to all years


In [3]:
# landuse, livestock에는 ExtrÃªme - Nord로 되어 있음
# 위에서 처리하기

landuse_clean.loc[landuse_clean['admin'] == 'ExtrÃªme - Nord', 'admin'] = 'Extrême - Nord'
livestock_clean.loc[livestock_clean['admin'] == 'ExtrÃªme - Nord', 'admin'] = 'Extrême - Nord'

In [4]:
# 병합 시작
result = base_df.copy()
print(f"Starting with base dataframe: {result.shape}")

# Climate 병합
result = result.merge(
    climate_clean,
    on=['country', 'admin', 'year'],
    how='left'
)
print(f"After merging Climate: {result.shape}")

# Livestock 병합
result = result.merge(
    livestock_clean,
    on=['country', 'admin', 'year'],
    how='left'
)
print(f"After merging Livestock: {result.shape}")

# Landuse 병합 (year 제외)
result = result.merge(
    landuse_clean,
    on=['country', 'admin'],
    how='left'
)
print(f"After merging Landuse: {result.shape}")
print(f"\n✓ All datasets merged successfully!")

# 최종 결과 확인
print("=== Final Merged Dataset ===")
print(f"Shape: {result.shape}")
print(f"\nColumns: {result.columns.tolist()}")
print(f"\nFirst few rows:")
display(result.head(10))

Starting with base dataframe: (488, 4)
After merging Climate: (488, 7)
After merging Livestock: (488, 15)
After merging Landuse: (488, 16)

✓ All datasets merged successfully!
=== Final Merged Dataset ===
Shape: (488, 16)

Columns: ['year', 'country', 'admin', 'infection_rate', 'max_temperature', 'min_temperature', 'avg_temperature', 'Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']

First few rows:


,year,country,admin,infection_rate,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse
0,2010,Burkina Faso,Boucle Du Mouhoun,0.585652,307.42114,297.76172,302.50760,0.0,25.000049,164.065700,0.0,34.153420,0.693863,21.774243,9.682960,0.076206
1,2010,Burkina Faso,Cascades,0.979432,304.47192,297.88160,300.27893,0.0,38.008688,96.864817,0.0,12.506484,0.008877,12.373391,3.865450,0.074387
2,2010,Burkina Faso,Centre,1.955671,307.73170,299.50660,302.80447,0.0,17.566445,303.908919,0.0,58.134766,0.019041,40.947445,6.821150,0.407856
3,2010,Burkina Faso,Centre-est,1.037736,307.41138,298.78003,302.61942,0.0,29.597071,203.094681,0.0,69.010943,0.027864,45.871831,18.292361,0.143950
4,2010,Burkina Faso,Centre-nord,0.379147,308.31274,297.82104,303.14368,0.0,28.843678,138.295650,0.0,65.134086,0.079677,48.346972,4.834737,0.120649
5,2010,Burkina Faso,Centre-ouest,1.622419,306.55396,299.12573,301.98790,0.0,4.144297,197.316045,0.0,32.718305,0.079125,25.059793,6.839734,0.121380
6,2010,Burkina Faso,Centre-sud,0.926784,306.74730,298.70190,302.07712,0.0,25.646751,248.165379,0.0,59.255066,0.005035,31.032994,13.007065,0.130138
7,2010,Burkina Faso,Est,0.957702,307.95923,297.30933,302.74704,0.0,22.987212,105.163249,0.0,31.965473,0.148155,19.576255,3.607876,0.056882
8,2010,Burkina Faso,Hauts-bassins,1.631543,305.89966,298.75854,301.59680,0.0,58.893784,129.289100,0.0,29.676095,0.009982,29.628014,11.868124,0.092157
9,2010,Burkina Faso,Nord,0.875657,307.55298,297.21558,302.34323,0.0,26.315218,177.275872,0.0,71.545219,0.542646,51.607829,10.399306,0.117150


## **disease data와 landuse, climate의 시간 차이로 인해 발생하는 결측치 분석**

### 각 데이터의 시간 범위
- disease : 2001-2023
- climate : 2001-2020
- livestock : 2001-2021

### 체크 포인트
- (climate 칼럼) 병합 disease의 2021/2022/2023 데이터에서 결측치일 가능성 체크
-  (livestock 칼럼) 병합 disease의 2022/2023 데이터에서 결측치일 가능성 체크

=> 만약 이 영향이 있다면 disease의 시간 범위를 2001-2020/2021로 하는 걸 고려해보기


In [3]:
disease_2021 = disease_df[disease_df['year']==2021]
disease_2021

,year,country,region_name,infection_rate


In [4]:
disease_2022 = disease_df[disease_df['year']==2022]
disease_2022

,year,country,region_name,infection_rate


In [7]:
disease_2023 = disease_df[disease_df['year']==2023]
disease_2023

,year,country,region_name,infection_rate
467,2023,Democratic Republic of the Congo,Bandundu,0.297359
468,2023,Democratic Republic of the Congo,Bas-Congo,0.844773
469,2023,Democratic Republic of the Congo,Equateur,0.829451
470,2023,Democratic Republic of the Congo,Kasai Occidental,0.475740
471,2023,Democratic Republic of the Congo,Kasai Oriental,0.655838
472,2023,Democratic Republic of the Congo,Katanga,1.307594
473,2023,Democratic Republic of the Congo,Kinshasa,0.422297
474,2023,Democratic Republic of the Congo,Maniema,0.394477
475,2023,Democratic Republic of the Congo,Nord-Kivu,0.806452
476,2023,Democratic Republic of the Congo,Orientale,2.533986


2021-2023 동안의 disease 데이터는
2023년도의 democratic republic of the congo가 유일(row 수 11개)

In [5]:
time_deletion = result['year']>2020
time_result = result[~time_deletion]

In [8]:
# check

time_result[time_deletion]

/var/folders/qk/g_3547n500j0qd7wpq7p90_r0000gn/T/ipykernel_89338/2824563463.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  time_result[time_deletion]


,year,country,admin,infection_rate,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse


In [6]:
time_result.shape

(477, 16)

### result의 시간 범위를 2001-2020으로 통일함
#### => time_result로 저장
2023년도의 democratic republic of the congo에 대한 11개의 데이터만 삭제됨

## **남은 데이터들에 대한 결측치 분석 마저 진행**

In [9]:
# 각 행의 결측치 비율
time_result['missing_ratio'] = time_result.isnull().sum(axis=1) / (len(time_result.columns) - 1)  # -1 for the new column

print("=== Row-wise Missing Data Analysis ===")
print(f"Rows with no missing values: {(time_result['missing_ratio'] == 0).sum()}")
print(f"Rows with < 25% missing: {(time_result['missing_ratio'] < 0.25).sum()}")
print(f"Rows with < 50% missing: {(time_result['missing_ratio'] < 0.50).sum()}")
print(f"Rows with >= 50% missing: {(time_result['missing_ratio'] >= 0.50).sum()}")


=== Row-wise Missing Data Analysis ===
Rows with no missing values: 473
Rows with < 25% missing: 477
Rows with < 50% missing: 477
Rows with >= 50% missing: 0


/var/folders/qk/g_3547n500j0qd7wpq7p90_r0000gn/T/ipykernel_89338/1959356534.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  time_result['missing_ratio'] = time_result.isnull().sum(axis=1) / (len(time_result.columns) - 1)  # -1 for the new column


In [10]:
time_result[time_result['missing_ratio'] > 0]

,year,country,admin,infection_rate,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse,missing_ratio
331,2017,Senegal,Dakar,0.353607,NaN,NaN,NaN,0.0,39.717247,71733.611842,0.000000,105.930420,11.841594,264.498047,3.441127,0.680112,0.2
345,2010,Senegal,Dakar,0.416233,NaN,NaN,NaN,0.0,36.085057,42253.598684,0.000000,85.050633,10.629105,214.727590,2.587541,0.680112,0.2
359,2005,Senegal,Dakar,0.682594,NaN,NaN,NaN,0.0,34.444387,28784.726974,0.000000,74.694092,10.915514,186.655530,2.390358,0.680112,0.2
411,2016,South Africa,Western Cape,11.335013,NaN,NaN,NaN,0.0,4.082979,75.611725,0.353363,1.713044,0.192779,21.323498,1.236929,0.036426,0.2


- (Cameroon, Extrême - Nord) : livestock, landuse 결측치
    - 2004,2011,2018년도 데이터 존재
- (Senegal,  Dakar) : climate 결측치
    - 2005, 2010, 2017 데이터 존재
- (South Africa, Western Cape) : climate 결측치
    - 2016 데이터 존재

### livestock, landuse 결측치 체크
- 둘 다 동일하게 africa shapefile로 zonal stats 했으니, shapefile 위주로 문제점 파악

- shapefile의 이름을 다시 봤는데 ExtrÃªme - Nord로 되어있음....
- disease region mapping에 이용한 파일은 shapefile과 동일한 홈페이지에서 제공하는 csv 파일의 지역명들을 활용한 것이기에 그 과정에서 오차가 발생한 듯

In [38]:
# climate, disease에는 Extrême - Nord로 되어 있음

set(climate_clean[climate_clean['country']=='Cameroon']['admin'])
#base_df[base_df['country']=='Cameroon']

{'Adamaoua',
 'Centre',
 'Est',
 'Extrême - Nord',
 'Littoral',
 'Nord',
 'Nord - Ouest',
 'Ouest',
 'Sud',
 'Sud - Ouest'}

In [ ]:
# landuse, livestock에는 ExtrÃªme - Nord로 되어 있음
# 위에서 처리하기

#landuse_clean.loc[landuse_clean['admin'] == 'ExtrÃªme - Nord', 'admin'] = 'Extrême - Nord'
#livestock_clean.loc[livestock_clean['admin'] == 'ExtrÃªme - Nord', 'admin'] = 'Extrême - Nord'

,country,admin,rate_landuse
324,Cameroon,ExtrÃªme - Nord,0.095491


In [49]:
# 데이터 병합 과정에서 위 코드 실행한 뒤 결측치 분석

# 각 행의 결측치 비율
time_result['missing_ratio'] = time_result.isnull().sum(axis=1) / (len(time_result.columns) - 1)  # -1 for the new column

print("=== Row-wise Missing Data Analysis ===")
print(f"Rows with no missing values: {(time_result['missing_ratio'] == 0).sum()}")
print(f"Rows with < 25% missing: {(time_result['missing_ratio'] < 0.25).sum()}")
print(f"Rows with < 50% missing: {(time_result['missing_ratio'] < 0.50).sum()}")
print(f"Rows with >= 50% missing: {(time_result['missing_ratio'] >= 0.50).sum()}")

time_result[time_result['missing_ratio'] > 0]

=== Row-wise Missing Data Analysis ===
Rows with no missing values: 473
Rows with < 25% missing: 477
Rows with < 50% missing: 477
Rows with >= 50% missing: 0


/var/folders/qk/g_3547n500j0qd7wpq7p90_r0000gn/T/ipykernel_56550/480238371.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  time_result['missing_ratio'] = time_result.isnull().sum(axis=1) / (len(time_result.columns) - 1)  # -1 for the new column


,year,country,admin,infection_rate,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse,missing_ratio
331,2017,Senegal,Dakar,0.353607,NaN,NaN,NaN,0.0,39.717247,71733.611842,0.000000,105.930420,11.841594,264.498047,3.441127,0.680112,0.2
345,2010,Senegal,Dakar,0.416233,NaN,NaN,NaN,0.0,36.085057,42253.598684,0.000000,85.050633,10.629105,214.727590,2.587541,0.680112,0.2
359,2005,Senegal,Dakar,0.682594,NaN,NaN,NaN,0.0,34.444387,28784.726974,0.000000,74.694092,10.915514,186.655530,2.390358,0.680112,0.2
411,2016,South Africa,Western Cape,11.335013,NaN,NaN,NaN,0.0,4.082979,75.611725,0.353363,1.713044,0.192779,21.323498,1.236929,0.036426,0.2


## climate 결측치 체크
### senegal에서 dakar

In [39]:
set(climate_df[climate_df['country']=='Senegal']['admin'])

{'Diourbel',
 'Fatick',
 'Kaffrine',
 'Kaolack',
 'Kedougou',
 'Kolda',
 'Louga',
 'Matam',
 'Saint louis',
 'Sedhiou',
 'Tambacounda',
 'Thies',
 'Ziguinchor'}

In [40]:
set(base_df[base_df['country']=='Senegal']['admin'])

{'Dakar',
 'Diourbel',
 'Fatick',
 'Kaffrine',
 'Kaolack',
 'Kedougou',
 'Kolda',
 'Louga',
 'Matam',
 'Saint louis',
 'Sedhiou',
 'Tambacounda',
 'Thies',
 'Ziguinchor'}

# **Climate 결측치 region 삭제**

(Senegal,  Dakar), (South Africa, Western Cape)

In [ ]:
region_deletion_1 = (time_result['country']=='Senegal') & (time_result['admin']== 'Dakar')
region_deletion_2 = (time_result['country']=='South Africa') & (time_result['admin']== 'Western Cape')
#time_result[region_deletion_1]
time_result[region_deletion_2]

,year,country,admin,infection_rate,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse,missing_ratio
411,2016,South Africa,Western Cape,11.335013,NaN,NaN,NaN,0.0,4.082979,75.611725,0.353363,1.713044,0.192779,21.323498,1.236929,0.036426,0.2


In [13]:
region_result_1 = time_result[~region_deletion_1]
region_result_1[region_deletion_1]

/var/folders/qk/g_3547n500j0qd7wpq7p90_r0000gn/T/ipykernel_89338/2737444183.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  region_result_1[region_deletion_1]


,year,country,admin,infection_rate,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse,missing_ratio


In [14]:
region_result_2 = region_result_1[~region_deletion_2]
region_result_2[region_deletion_2]

/var/folders/qk/g_3547n500j0qd7wpq7p90_r0000gn/T/ipykernel_89338/2645004542.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  region_result_2 = region_result_1[~region_deletion_2]
/var/folders/qk/g_3547n500j0qd7wpq7p90_r0000gn/T/ipykernel_89338/2645004542.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  region_result_2[region_deletion_2]


,year,country,admin,infection_rate,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse,missing_ratio


In [15]:
final_result = region_result_2

## **결측치 없앤 최종 데이터셋 csv 변환하기**

In [16]:
output_filename = '../../../data/processed/cleaned_disease_dataset_2015landuse.csv'
final_result.to_csv(output_filename, index=False)